In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.compose import ColumnTransformer

In [ ]:
df=pd.read_csv('train.csv',usecols=['Age','Fare','Survived'])

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
X=df.iloc[:,1:]
y=df.iloc[:,0]

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2, random_state=42)

In [ ]:
X_train.head()

In [ ]:
clf=DecisionTreeClassifier()

In [ ]:
clf.fit(X_train,y_train)
y_pred=clf.predict(X_test)

In [ ]:
accuracy_score(y_test,y_pred)

In [ ]:
np.mean(cross_val_score(DecisionTreeClassifier(),X,y,cv=10,scoring='accuracy'))

In [ ]:
kbin_age=KBinsDiscretizer(n_bins=15,encode='ordinal',strategy='quantile')
kbin_fare=KBinsDiscretizer(n_bins=15,encode='ordinal',strategy='quantile')

In [ ]:
trf=ColumnTransformer([
    ('first',kbin_age,[0]),
    ('second',kbin_fare,[1])
])

In [ ]:
X_train_trf=trf.fit_transform(X_train)
X_test_trf=trf.transform(X_test)

In [ ]:
trf.named_transformers_['first'].bin_edges_

In [ ]:
trf.named_transformers_['second'].bin_edges_

In [ ]:
output=pd.DataFrame({
    'age':X_train['Age'],
    'age_trf':X_train_trf[:,0],
    'fare':X_train['Fare'],
    'fare_trf':X_train_trf[:,1]
})

In [ ]:
output['age_labels'] = pd.cut(x=X_train['Age'],
                              bins=trf.named_transformers_['first'].bin_edges_[0].tolist())
output['fare_labels'] = pd.cut(x=X_train['Fare'],
                               bins=trf.named_transformers_['second'].bin_edges_[0].tolist())

In [ ]:
output.sample(5)

In [ ]:
clf=DecisionTreeClassifier()
clf.fit(X_train_trf,y_train)
y_pred2=clf.predict(X_test_trf)

In [ ]:
accuracy_score(y_test,y_pred2)

In [ ]:
X_trf = trf.fit_transform(X)
np.mean(cross_val_score(DecisionTreeClassifier(),X,y,cv=10,scoring='accuracy'))

Custom Function For Binarization

In [ ]:
def discretize(bins,strategy):
 kbin_age=KBinsDiscretizer(n_bins=bins,encode='ordinal',strategy=strategy)
 kbin_fare = KBinsDiscretizer(n_bins=bins,encode='ordinal',strategy=strategy)
 trf=ColumnTransformer([('first',kbin_age,[0]),
                          ('second',kbin_fare,[1])])
 X_trf = trf.fit_transform(X)
 print(np.mean(cross_val_score(DecisionTreeClassifier(),X,y,cv=10,scoring='accuracy')))
 plt.figure(figsize=(14,4))
 plt.subplot(121)
 plt.hist(X['Age'])
 plt.title("Before")

 plt.subplot(122)
 plt.hist(X_trf[:,0],color='red')
 plt.title("After")
 plt.show()

 plt.figure(figsize=(14,4))
 plt.subplot(121)
 plt.hist(X['Fare'])
 plt.title("Before")

 plt.subplot(122)
 plt.hist(X_trf[:,1],color='red')
 plt.title("Fare")

 plt.show()

In [ ]:
discretize(5,'kmeans')